# Step 1 - Select games and candidate positions

Step 1 - select games and candidate positions from the pinned Lichess file (reads step 0's outputs).

Notebook: open step1_select_positions.ipynb and Run All from the project folder inside .venv.

Pass 1  verifies the raw SHA-256, then streams the 30 GB file to apply game-level criteria and write candidate_games.pgn
        (cached: later runs skip the filter stream if the fingerprint is unchanged) while checking duplicate IDs.
Pass 2  replays the surviving games (python-chess), finds eligible good / bad moves from the Lichess evals, picks one
        position per game with a per-game seeded shuffle, writes the full selected tables + manifest; pilot games are not reserved until requested later.
All scientific settings come from config.py. The Stockfish re-check is included as Step 2 below.

Save the notebook (Ctrl+S) before running so the recorded source hash matches the code being executed. Use **Run All** for Step 1 only while `RUN_STOCKFISH = False`. Step 1 keeps the full selected data and does not reserve pilot games by default; Step 2 can re-check this unsplit pool before pilot sampling. A full run processes the large compressed monthly file; test settings at the top use a separate `_TEST` output folder.


## Parameters


In [1]:
import array, datetime, hashlib, io, json, math, platform, random, re, subprocess, sys, time
from importlib import metadata
from pathlib import Path
import numpy as np, pandas as pd, zstandard, chess, chess.pgn
import config

def notebook_code_sha256(stage):
    path = Path.cwd() / "step1_select_positions.ipynb"
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source = "\n".join("".join(cell["source"]) for cell in notebook["cells"]
                         if stage in cell.get("metadata", {}).get("tags", []))
    return hashlib.sha256(source.encode("utf-8")).hexdigest()

MAX_GAMES = None          # None = whole file. Test only, e.g. 2_000_000 (outputs then go to a separate *_TEST folder)
RAW_OVERRIDE = None       # test only: path to a small .pgn.zst
DUP_CHECK_ALL = True      # True checks every game (~1.5 GB peak RAM); False checks only games that pass the filters
FORCE_PASS1 = False       # True = ignore the cached candidate_games.pgn
CODE_VERSION = "step1-v2"  # bump when you change the logic below

PROJECT = Path.cwd()
MONTH = config.MONTH
RAW = Path(RAW_OVERRIDE) if RAW_OVERRIDE else PROJECT / "data" / "raw" / config.LICHESS_FILE
S0 = PROJECT / "outputs" / "step0_dataset" / MONTH
OUT = PROJECT / "outputs" / "step1_dataset" / (MONTH + ("_TEST" if (MAX_GAMES or RAW_OVERRIDE) else ""))
OUT.mkdir(parents=True, exist_ok=True)
assert config.POSITIONS_PER_GAME == {"good": 1, "bad": 1}, "this script supports one good + one bad position per game"


def git(*a):
    try: return subprocess.run(["git", "-C", str(PROJECT), *a], check=True, capture_output=True, text=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError): return None


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while blk := f.read(1 << 24): h.update(blk)
    return h.hexdigest()


STEP1_SOURCE_SHA256 = notebook_code_sha256("step1")
PASS1_SOURCE_SHA256 = notebook_code_sha256("step1_pass1")


## 0. Read Step 0 and resolve the rating threshold


In [2]:
dm = json.loads((S0 / "download_manifest.json").read_text(encoding="utf-8"))
pm = json.loads((S0 / "profile_manifest.json").read_text(encoding="utf-8"))
if not RAW.is_file():
    raise FileNotFoundError(f"Raw PGN file not found: {RAW}")
raw_bytes = RAW.stat().st_size
assert not pm.get("partial", True), "step 0 profile is PARTIAL - run step 0 fully first"
if not RAW_OVERRIDE:
    assert raw_bytes == dm["bytes"], "raw file size differs from download_manifest.json"
    assert dm.get("matches_official") in (True, None), "raw file checksum did not match the official one"
print("Verifying raw-file SHA-256 (reads the compressed file once)...", flush=True)
raw_sha256 = sha256_file(RAW)
if not RAW_OVERRIDE:
    assert raw_sha256 == dm["sha256"], "raw file SHA-256 differs from download_manifest.json"

elo = pd.read_csv(S0 / "tables" / "elo_exact.csv")
pool = (elo.white + elo.black).values                     # same definition as step 0's percentile table
cum = np.cumsum(pool) / pool.sum()
ELO_THR = int(elo.elo.values[np.searchsorted(cum, config.ELO_PERCENTILE / 100)])
print(f"P{config.ELO_PERCENTILE} of player ratings in {MONTH} = {ELO_THR}  -> both players must be >= {ELO_THR}")


Verifying raw-file SHA-256 (reads the compressed file once)...
P99.0 of player ratings in 2026-10 = 2556  -> both players must be >= 2556


## 1. Pass 1: Stream and filter games


In [3]:
SEP = b"\n\n[Event "
TC = re.compile(rb"(\d+)\+(\d+)")
COMMENT = re.compile(rb"\{[^}]*\}")


def iter_games(path, block=1 << 26):
    """yield each game as bytes WITHOUT the leading '[Event ' (so every tag line starts after a newline)."""
    dctx = zstandard.ZstdDecompressor(max_window_size=2**31)
    with open(path, "rb") as f, dctx.stream_reader(f, read_size=1 << 22) as r:
        buf, first = b"", True
        while blk := r.read(block):
            buf += blk
            if first: buf = buf[len(b"[Event "):] if buf.startswith(b"[Event ") else buf; first = False
            parts = buf.split(SEP); buf = parts.pop()
            yield from parts
        if buf.strip(): yield buf.rstrip()


def tag(h, key):
    i = h.find(key)
    if i < 0: return None
    i += len(key); return h[i:h.find(b'"', i)]


def to_elo(v):
    try:
        x = int(v); return x if 0 < x < 4000 else None
    except (TypeError, ValueError): return None


def plies_of(mt):
    n = 0
    for t in COMMENT.sub(b"", mt).split():
        if t[:1] == b"$" or t in (b"1-0", b"0-1", b"1/2-1/2", b"*") or re.fullmatch(rb"\d+\.+", t): continue
        n += 1
    return n


STEPS = ["all games", f"both players >= {ELO_THR} (P{config.ELO_PERCENTILE})", f"rating gap <= {config.MAX_RATING_GAP}",
         f"rapid or slower (est. >= {config.MIN_EST_SECONDS}s)", "normal termination", "no bot account",
         f"at least {config.MIN_PLIES} plies", "Lichess evals present"]
pass1_fp = hashlib.sha256(json.dumps(dict(sha=raw_sha256, thr=ELO_THR, gap=config.MAX_RATING_GAP, est=config.MIN_EST_SECONDS,
          plies=config.MIN_PLIES, normal=config.REQUIRE_NORMAL_TERMINATION, bots=config.EXCLUDE_BOTS,
          evals=config.REQUIRE_LICHESS_EVALS, max_games=MAX_GAMES, raw=str(RAW_OVERRIDE), dup=DUP_CHECK_ALL,
          code=CODE_VERSION, pass1_source_sha256=PASS1_SOURCE_SHA256), sort_keys=True).encode()).hexdigest()
P1 = OUT / "pass1_manifest.json"; CAND = OUT / "candidate_games.pgn"

if CAND.exists() and P1.exists() and json.loads(P1.read_text())["fingerprint"] == pass1_fp and not FORCE_PASS1:
    p1 = json.loads(P1.read_text()); print("Pass 1 cached (fingerprint unchanged) - skipping the 30 GB stream.")
else:
    P1.unlink(missing_ok=True)
    cnt = [0] * len(STEPS); ids = array.array("Q"); t0 = time.time()
    with open(CAND, "wb") as fout:
        for g in iter_games(RAW):
            cnt[0] += 1
            if MAX_GAMES and cnt[0] > MAX_GAMES: cnt[0] -= 1; break
            if cnt[0] % 5_000_000 == 0:
                print(f"  {cnt[0]:>11,} games  kept {cnt[-1]}  {time.time() - t0:5.0f}s", flush=True)
            hdr = g[:g.find(b"\n\n")]
            site = tag(hdr, b'\n[Site "')
            game_id = (site or b"").rsplit(b"/", 1)[-1]
            if len(game_id) != 8:
                raise ValueError(f"Game {cnt[0]:,} has a missing or unexpected Lichess Site ID: {site!r}")
            game_id_value = int.from_bytes(game_id, "big")
            if DUP_CHECK_ALL:
                ids.append(game_id_value)
            w, b = to_elo(tag(hdr, b'\n[WhiteElo "')), to_elo(tag(hdr, b'\n[BlackElo "'))
            if w is None or b is None or min(w, b) < ELO_THR: continue
            cnt[1] += 1
            if abs(w - b) > config.MAX_RATING_GAP: continue
            cnt[2] += 1
            m = TC.fullmatch(tag(hdr, b'\n[TimeControl "') or b"")
            if not m or int(m[1]) + 40 * int(m[2]) < config.MIN_EST_SECONDS: continue
            cnt[3] += 1
            if config.REQUIRE_NORMAL_TERMINATION and tag(hdr, b'\n[Termination "') != b"Normal": continue
            cnt[4] += 1
            if config.EXCLUDE_BOTS and (b'\n[WhiteTitle "BOT"]' in hdr or b'\n[BlackTitle "BOT"]' in hdr): continue
            cnt[5] += 1
            mt = g[len(hdr) + 2:]
            if plies_of(mt) < config.MIN_PLIES: continue
            cnt[6] += 1
            if config.REQUIRE_LICHESS_EVALS and b"[%eval" not in mt: continue
            cnt[7] += 1
            if not DUP_CHECK_ALL:
                ids.append(game_id_value)
            fout.write(b"[Event " + g + b"\n\n")
    if ids:
        a = np.sort(np.frombuffer(ids, dtype=np.uint64))
        duplicate_count, n_ids = int((a[1:] == a[:-1]).sum()), len(a)
        del a, ids
    else:
        duplicate_count, n_ids = 0, 0
    p1 = dict(fingerprint=pass1_fp, raw_sha256=raw_sha256, funnel=dict(zip(STEPS, cnt)),
              duplicate_ids_in_whole_file=duplicate_count if DUP_CHECK_ALL else None,
              duplicate_ids_in_survivors=duplicate_count if not DUP_CHECK_ALL else None, ids_checked=n_ids,
              seconds=round(time.time() - t0))
    P1.write_text(json.dumps(p1, indent=2))
print(pd.Series(p1["funnel"], name="games").to_frame())
if DUP_CHECK_ALL:
    print("duplicate game IDs in the whole file:", p1["duplicate_ids_in_whole_file"])
else:
    print("duplicate game IDs among survivors:", p1["duplicate_ids_in_survivors"])


    5,000,000 games  kept 25     38s
   10,000,000 games  kept 33     74s
   15,000,000 games  kept 44    111s
   20,000,000 games  kept 65    146s
   25,000,000 games  kept 80    179s
   30,000,000 games  kept 96    212s
   35,000,000 games  kept 118    246s
   40,000,000 games  kept 127    282s
   45,000,000 games  kept 148    316s
   50,000,000 games  kept 168    352s
   55,000,000 games  kept 184    387s
   60,000,000 games  kept 198    422s
   65,000,000 games  kept 216    458s
   70,000,000 games  kept 223    492s
   75,000,000 games  kept 230    527s
   80,000,000 games  kept 243    576s
   85,000,000 games  kept 253    635s
   90,000,000 games  kept 262    693s
   95,000,000 games  kept 271    750s
                                   games
all games                       95395628
both players >= 2556 (P99.0)      704444
rating gap <= 100                 497608
rapid or slower (est. >= 480s)     11670
normal termination                 10706
no bot account                       4

## 2. Pass 2: Replay games and select positions


In [4]:
def winprob(cp): return 0.5 + 0.5 * (2 / (1 + math.exp(-0.00368208 * cp)) - 1)       # Lichess win% formula, 0-1


def analyse(game):
    """all candidate moves (full moves FULLMOVE_MIN..MAX) with Lichess evals before/after; mover's point of view."""
    nodes = list(game.mainline()); cps = []
    for n in nodes:
        e = n.eval(); cps.append(None if e is None else e.white().score())            # None for mate scores or missing eval
    board, out = game.board(), []
    for i, n in enumerate(nodes):
        if i >= 1 and config.FULLMOVE_MIN <= board.fullmove_number <= config.FULLMOVE_MAX:
            b, a = cps[i - 1], cps[i]
            if b is not None and a is not None and abs(b) <= config.MAX_ABS_EVAL_CP and not board.is_check() \
                    and board.legal_moves.count() >= 2:
                s = 1 if board.turn else -1
                loss = max(0, s * b - s * a)
                out.append(dict(ply=i + 1, fullmove=board.fullmove_number, side="white" if board.turn else "black",
                                fen=board.fen(), pos_key=" ".join(board.fen().split()[:4]), move_san=board.san(n.move),
                                move_uci=n.move.uci(), eval_before_cp=s * b, eval_after_cp=s * a, cp_loss=loss,
                                winprob_drop=max(0.0, winprob(s * b) - winprob(s * a))))
        board.push(n.move)
    return out


games = {}
with open(CAND, encoding="utf-8") as f:
    while (g := chess.pgn.read_game(f)) is not None:
        h = g.headers; gid = h["Site"].rsplit("/", 1)[-1]
        if gid in games:
            raise ValueError(f"duplicate candidate game ID {gid!r} in candidate_games.pgn")
        m = TC.fullmatch(h.get("TimeControl", "").encode())
        games[gid] = dict(game_id=gid, white=h.get("White", ""), black=h.get("Black", ""), white_elo=int(h["WhiteElo"]), black_elo=int(h["BlackElo"]), time_control=h["TimeControl"],
                          result=h["Result"], eco=h.get("ECO", ""), date=h.get("UTCDate", ""), moves=analyse(g))
print(f"{len(games)} candidate games replayed (duplicate IDs among them: 0)")
assert len(games) == p1["funnel"][STEPS[-1]], "number of replayed games differs from pass 1 funnel (duplicate game IDs?)"

BAD = (lambda d: d["cp_loss"] >= config.BAD_MIN_CP_LOSS) if config.BAD_DEFINITION == "cp" else \
      (lambda d: d["winprob_drop"] >= config.BAD_MIN_WINPROB_DROP)
GOOD = lambda d: d["cp_loss"] <= config.GOOD_MAX_CP_LOSS


def pick(cands, gid, tag_, used):
    c = sorted(cands, key=lambda d: d["ply"])
    random.Random(f"{config.SEED_SELECTION}:{gid}:{tag_}").shuffle(c)                 # seeded per game: independent of order
    for d in c:
        if d["pos_key"] not in used: used.add(d["pos_key"]); return d
    return None


used, rows, summ = set(), [], []
for gid in sorted(games):                                                             # fixed order -> reproducible "position not already used"
    G = games[gid]; mv = G["moves"]
    good, bad_cp, bad_wp = [d for d in mv if GOOD(d)], [d for d in mv if d["cp_loss"] >= config.BAD_MIN_CP_LOSS], \
                           [d for d in mv if d["winprob_drop"] >= config.BAD_MIN_WINPROB_DROP]
    pg = pick(good, gid, "good", used)
    pb = pick([d for d in mv if BAD(d)], gid, "bad", used) if (pg or not config.BAD_ONLY_IN_GAMES_WITH_GOOD) else None
    summ.append(dict(game_id=gid, n_eligible_good=len(good), n_bad_cp=len(bad_cp), n_bad_winprob=len(bad_wp),
                     has_good_position=int(pg is not None), has_bad_position=int(pb is not None)))
    for kind, p in (("good", pg), ("bad", pb)):
        if p: rows.append(dict(game_id=gid, kind=kind, **{k: v for k, v in p.items() if k != "pos_key"},
                               **{k: G[k] for k in ("white", "black", "white_elo", "black_elo", "time_control", "result", "eco", "date")}))
summ = pd.DataFrame(summ); pos = pd.DataFrame(rows)
if pos.empty:
    raise RuntimeError("No eligible positions were selected; check the input and configured thresholds.")

# optional random subsample of games (GAMES_TARGET), then the pilot split (game level, so no leakage between the sets)
good_games = sorted(pos[pos.kind == "good"].game_id)
if config.GAMES_TARGET is not None and config.GAMES_TARGET < len(good_games):
    keep = set(random.Random(f"{config.SEED_SELECTION}:sample").sample(good_games, config.GAMES_TARGET))
    pos = pos[pos.game_id.isin(keep)]; good_games = sorted(keep)
MAKE_PILOT_SPLIT = False  # Later, set True to sample PILOT_N whole games out of the selected data.
if MAKE_PILOT_SPLIT:
    pilot = set(random.Random(f"{config.SEED_PILOT_SPLIT}:pilot").sample(good_games, min(config.PILOT_N, len(good_games))))
else:
    pilot = set()
pos["is_pilot"] = pos.game_id.isin(pilot).astype(int)
pos = pos.sort_values(["kind", "game_id"]).reset_index(drop=True)
pos.insert(0, "position_id", [f"{r.kind[0].upper()}{i:04d}" for i, r in pos.iterrows()])


271 candidate games replayed (duplicate IDs among them: 0)


## 3. Write tables and calculate KPIs


In [5]:
def save(name, df): df.to_csv(OUT / name, index=False, lineterminator="\n", float_format="%.10g")


funnel = pd.DataFrame({"step": STEPS, "games": [p1["funnel"][s] for s in STEPS]})
nG = int(summ.has_good_position.sum())
extra = [("has a good-move position (<= %d cp loss, not decided, board checks)" % config.GOOD_MAX_CP_LOSS, nG),
         (f"... of which pilot (excluded from final)", len(pilot)), ("final games (good positions)", nG - len(pilot))]
funnel = pd.concat([funnel, pd.DataFrame(extra, columns=["step", "games"])], ignore_index=True)
funnel["share_of_previous"] = funnel.games / funnel.games.shift(1)
funnel.loc[funnel.index[-2:], "share_of_previous"] = None   # pilot / final rows are splits, not filters
save("funnel_game_level.csv", funnel)

sg = summ[summ.has_good_position == 1]
bad_counts = pd.DataFrame({"definition": [f"loss >= {config.BAD_MIN_CP_LOSS} cp", f"win-prob drop >= {config.BAD_MIN_WINPROB_DROP:.2f}"],
    "games_with_good_position_and_such_a_move": [int((sg.n_bad_cp > 0).sum()), int((sg.n_bad_winprob > 0).sum())],
    "games_in_all_candidates": [int((summ.n_bad_cp > 0).sum()), int((summ.n_bad_winprob > 0).sum())],
    "bad_moves_total": [int(summ.n_bad_cp.sum()), int(summ.n_bad_winprob.sum())]})
bad_counts["advisor_threshold_40_reached"] = bad_counts.games_with_good_position_and_such_a_move >= 40
save("bad_move_counts.csv", bad_counts); save("game_summary.csv", summ); save("positions_all.csv", pos)
save("positions_pilot.csv", pos[pos.is_pilot == 1]); save("positions_final.csv", pos[pos.is_pilot == 0])

fin = pos[pos.is_pilot == 0].assign(min_elo=lambda d: d[["white_elo", "black_elo"]].min(axis=1),
                                    rating_gap=lambda d: (d.white_elo - d.black_elo).abs(), abs_eval_before=lambda d: d.eval_before_cp.abs(),
                                    est_seconds=lambda d: d.time_control.map(lambda t: int(t.split("+")[0]) + 40 * int(t.split("+")[1])))
kpi = fin.groupby("kind")[["min_elo", "rating_gap", "fullmove", "abs_eval_before", "cp_loss", "winprob_drop", "est_seconds"]].describe().T
kpi.index = ["_".join(i) for i in kpi.index]; save("kpis_final_positions.csv", kpi.reset_index().rename(columns={"index": "stat"}))
side = fin.groupby(["kind", "side"]).size().rename("positions").reset_index(); save("side_to_move.csv", side)
print(funnel.to_string(index=False)); print(); print(bad_counts.to_string(index=False)); print(); print(side.to_string(index=False))


                                                               step    games  share_of_previous
                                                          all games 95395628                NaN
                                       both players >= 2556 (P99.0)   704444           0.007384
                                                  rating gap <= 100   497608           0.706384
                                     rapid or slower (est. >= 480s)    11670           0.023452
                                                 normal termination    10706           0.917395
                                                     no bot account      411           0.038390
                                                  at least 60 plies      317           0.771290
                                              Lichess evals present      271           0.854890
has a good-move position (<= 30 cp loss, not decided, board checks)      270           0.996310
                           ... of which 

## 4. Write the reproducibility manifest


In [6]:
outs = sorted(p.name for p in OUT.glob("*.csv"))
h = hashlib.sha256()
for n in outs: h.update(n.encode()); h.update((OUT / n).read_bytes())
status = git("status", "--porcelain")
manifest = dict(step="step1_selection", code_version=CODE_VERSION, month=MONTH, raw_sha256=raw_sha256, raw_bytes=raw_bytes,
    step0_raw_sha256=dm["sha256"], step1_source_sha256=STEP1_SOURCE_SHA256,
    step0_results_sha256=pm.get("results_sha256"), rating_percentile=config.ELO_PERCENTILE, rating_threshold=ELO_THR,
    run_parameters=dict(max_games=MAX_GAMES, raw_override_used=bool(RAW_OVERRIDE), duplicate_check_all=DUP_CHECK_ALL,
                        make_pilot_split=MAKE_PILOT_SPLIT, pilot_n_games=config.PILOT_N if MAKE_PILOT_SPLIT else 0),
    funnel=p1["funnel"], duplicate_ids_in_whole_file=p1["duplicate_ids_in_whole_file"],
    duplicate_ids_in_survivors=p1["duplicate_ids_in_survivors"], ids_checked=p1["ids_checked"],
    games_final=nG - len(pilot), pilot_games=sorted(pilot), pilot_split_complete=MAKE_PILOT_SPLIT,
    timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), python=sys.version.split()[0],
    platform=platform.platform(), packages={p: metadata.version(p) for p in ("chess", "zstandard", "numpy", "pandas")},
    config=config.config_snapshot(), git_commit=git("rev-parse", "HEAD"), git_uncommitted_changes=bool(status),
    candidate_games_sha256=sha256_file(CAND), hashed_tables=outs, results_sha256=h.hexdigest(), partial=bool(MAX_GAMES or RAW_OVERRIDE))
(OUT / "selection_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("results_sha256:", manifest["results_sha256"], "| uncommitted changes:", manifest["git_uncommitted_changes"])


results_sha256: 574c6192f709490cfd136ccd482b5936e9c50a28a943b6d784762c4da8313075 | uncommitted changes: True


## Step 2 - Stockfish re-check

Re-check Step 1's final positions with the configured Stockfish engine.

Run from the project folder using the .venv kernel.
Interrupted runs resume from analysed_all.jsonl when their input, engine, config,
and source fingerprint still match.


This section runs the Stockfish engine when its final code cell is executed. The saved `RUN_STOCKFISH` switch is currently `True`; do not use **Run All** in this state. For a Step 1 **Run All**, set it to `False` first. To launch the re-check manually, ensure Step 1 is complete and Stockfish is configured, set the switch to `True`, save the notebook, and execute only the final cell. The source fingerprint is computed from the saved notebook. For a later session, run the Step 2 setup and function cells, then the final cell.


### Step 2 setup and helpers


In [11]:
import datetime
import hashlib
import json
import math
import platform
import shutil
import subprocess
import sys
from collections import Counter
from importlib import metadata
from pathlib import Path

import chess
import chess.engine
import pandas as pd

import config

def notebook_code_sha256(stage):
    path = Path.cwd() / "step1_select_positions.ipynb"
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source = "\n".join("".join(cell["source"]) for cell in notebook["cells"]
                         if stage in cell.get("metadata", {}).get("tags", []))
    return hashlib.sha256(source.encode("utf-8")).hexdigest()

CODE_VERSION = "step2-v1"
PROJECT = Path.cwd()
MONTH = config.MONTH
STEP1_OUT = PROJECT / "outputs" / "step1_dataset" / MONTH
IN = STEP1_OUT / "positions_final.csv"
STEP1_MANIFEST = STEP1_OUT / "selection_manifest.json"
OUT = PROJECT / "outputs" / "stockfish_recheck" / MONTH

ALL_PATH = OUT / "analysed_all.jsonl"
STATE_PATH = OUT / "resume_state.json"
KEPT_PATH = OUT / "positions_stockfish.csv"
REJECTED_PATH = OUT / "positions_rejected.csv"
KPI_PATH = OUT / "kpis.json"
MANIFEST_PATH = OUT / "stockfish_manifest.json"


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while chunk := stream.read(1 << 24):
            digest.update(chunk)
    return digest.hexdigest()


def git(*args):
    try:
        return subprocess.run(
            ["git", "-C", str(PROJECT), *args],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


def find_stockfish():
    path = Path(config.STOCKFISH_PATH)
    path = path if path.is_absolute() else PROJECT / path
    if path.is_file():
        return path.resolve()
    found = shutil.which("stockfish")
    if found:
        return Path(found).resolve()
    raise FileNotFoundError(
        f"Stockfish not found at {path} or on PATH; set config.STOCKFISH_PATH."
    )


def write_json_atomic(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary.replace(path)


def winprob(cp):
    return 0.5 + 0.5 * (2 / (1 + math.exp(-0.00368208 * cp)) - 1)


def label_for(cp_loss, winprob_drop):
    if cp_loss <= config.GOOD_MAX_CP_LOSS:
        return "good"
    if config.BAD_DEFINITION == "cp":
        return "bad" if cp_loss >= config.BAD_MIN_CP_LOSS else None
    if config.BAD_DEFINITION == "winprob":
        return "bad" if winprob_drop >= config.BAD_MIN_WINPROB_DROP else None
    raise ValueError(f"Unsupported BAD_DEFINITION: {config.BAD_DEFINITION!r}")


def select_info(info):
    return info[0] if isinstance(info, list) else info


def pv_san(board, pv):
    copy = board.copy()
    moves = []
    for move in pv[: config.PV_MAX_PLIES]:
        moves.append(copy.san(move))
        copy.push(move)
    return moves


def analyse_position(engine, candidate):
    position_id = str(candidate["position_id"])
    board = chess.Board(str(candidate["fen"]))
    move = chess.Move.from_uci(str(candidate["move_uci"]))
    if not board.is_legal(move):
        raise ValueError(f"{position_id}: selected move is illegal in its FEN")

    mover = board.turn
    before = select_info(
        engine.analyse(
            board,
            chess.engine.Limit(depth=config.STOCKFISH_DEPTH),
            multipv=config.STOCKFISH_MULTIPV,
            game=position_id + "_before",
        )
    )
    after_board = board.copy()
    after_board.push(move)
    after = select_info(
        engine.analyse(
            after_board,
            chess.engine.Limit(depth=config.STOCKFISH_DEPTH),
            multipv=config.STOCKFISH_MULTIPV,
            game=position_id + "_after",
        )
    )

    best_pv = before.get("pv", [])
    if not best_pv:
        raise RuntimeError(f"{position_id}: Stockfish returned no principal variation")

    before_score, after_score = before["score"], after["score"]
    reasons = []
    row = dict(candidate)
    row.update(
        engine_best_uci=best_pv[0].uci(),
        engine_best_san=board.san(best_pv[0]),
        engine_best_pv_san=pv_san(board, best_pv),
        human_move_is_engine_best=(best_pv[0] == move),
        rejected_reasons=reasons,
        depth_reached=after.get("depth"),
    )
    if before_score.is_mate() or after_score.is_mate():
        reasons.append("mate_score")
        row.update(stockfish_label=None, stockfish_cp_loss=None, stockfish_winprob_drop=None)
        return row

    before_cp = before_score.pov(mover).score()
    after_cp = after_score.pov(mover).score()
    before_white_cp = before_score.white().score()
    if before_cp is None or after_cp is None or before_white_cp is None:
        raise RuntimeError(f"{position_id}: Stockfish returned a non-mate score without centipawns")

    cp_loss = max(0, before_cp - after_cp)
    wp_drop = max(0.0, winprob(before_cp) - winprob(after_cp))
    stockfish_label = label_for(cp_loss, wp_drop)
    if abs(before_white_cp) > config.MAX_ABS_EVAL_CP:
        reasons.append("already_decided")
    if stockfish_label != candidate["kind"]:
        reasons.append("label_not_confirmed_by_stockfish")

    row.update(
        eval_before_cp=before_white_cp,
        eval_after_cp=after_score.white().score(),
        cp_loss=cp_loss,
        winprob_drop=round(wp_drop, 1),
        stockfish_label=stockfish_label,
        stockfish_cp_loss=cp_loss,
        stockfish_winprob_drop=round(wp_drop, 1),
        pv_after_san=pv_san(after_board, after.get("pv", [])),
    )
    return row


def jsonable(value):
    if isinstance(value, (list, tuple, dict)):
        return value
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def csv_rows(rows):
    output = []
    for row in rows:
        converted = {key: jsonable(value) for key, value in row.items()}
        for key in ("engine_best_pv_san", "pv_after_san", "rejected_reasons"):
            if key in converted:
                converted[key] = json.dumps(converted[key], ensure_ascii=False)
        output.append(converted)
    return output


### Step 2 analysis


In [12]:
def run_stockfish_recheck():
    OUT.mkdir(parents=True, exist_ok=True)
    STEP2_SOURCE_SHA256 = notebook_code_sha256("step2")
    if config.BAD_DEFINITION not in {"cp", "winprob"}:
        raise ValueError(f"Unsupported BAD_DEFINITION: {config.BAD_DEFINITION!r}")
    if not IN.is_file():
        raise FileNotFoundError(f"Step 1 output not found: {IN}; run Step 1 first.")
    if not STEP1_MANIFEST.is_file():
        raise FileNotFoundError(f"Step 1 manifest not found: {STEP1_MANIFEST}")
    step1_manifest = json.loads(STEP1_MANIFEST.read_text(encoding="utf-8"))
    if step1_manifest.get("partial"):
        raise ValueError("Step 1 output is partial; complete Step 1 before Stockfish re-check.")
    current_step1_source_sha256 = notebook_code_sha256("step1")
    if step1_manifest.get("step1_source_sha256") != current_step1_source_sha256:
        raise ValueError("Step 1 notebook source changed since its outputs were created; rerun Step 1 before Step 2.")

    input_sha256 = sha256_file(IN)
    step1_tables = step1_manifest.get("hashed_tables")
    if not isinstance(step1_tables, list) or "positions_final.csv" not in step1_tables:
        raise ValueError("Step 1 manifest does not list positions_final.csv in hashed_tables.")
    step1_hash = hashlib.sha256()
    for name in step1_tables:
        table_path = STEP1_OUT / name
        if not table_path.is_file():
            raise FileNotFoundError(f"Step 1 manifest table is missing: {table_path}")
        step1_hash.update(name.encode("utf-8"))
        step1_hash.update(table_path.read_bytes())
    if step1_hash.hexdigest() != step1_manifest.get("results_sha256"):
        raise ValueError("Step 1 tables do not match selection_manifest.json results_sha256.")

    candidates = [
        {key: jsonable(value) for key, value in row.items()}
        for row in pd.read_csv(IN).to_dict(orient="records")
    ]
    position_ids = [str(candidate["position_id"]) for candidate in candidates]
    if len(position_ids) != len(set(position_ids)):
        raise ValueError("positions_final.csv contains duplicate position_id values.")
    if "is_pilot" in (candidates[0] if candidates else {}) and any(
        bool(candidate["is_pilot"]) for candidate in candidates
    ):
        raise ValueError("positions_final.csv unexpectedly contains pilot positions.")

    stockfish = find_stockfish()
    engine_sha256 = sha256_file(stockfish)
    git_commit = git("rev-parse", "HEAD")
    git_status = git("status", "--porcelain")
    fingerprint_data = dict(
        code_version=CODE_VERSION,
        step2_source_sha256=STEP2_SOURCE_SHA256,
        input_sha256=input_sha256,
        engine_sha256=engine_sha256,
        config=config.config_snapshot(),
    )
    fingerprint = hashlib.sha256(
        json.dumps(fingerprint_data, sort_keys=True).encode("utf-8")
    ).hexdigest()

    if ALL_PATH.exists() and not STATE_PATH.exists():
        raise RuntimeError(
            "analysed_all.jsonl exists without resume_state.json; "
            "preserve or move the incomplete Step 2 output before restarting."
        )
    if STATE_PATH.exists():
        state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
        if state.get("fingerprint") != fingerprint:
            raise RuntimeError(
                "Step 2 resume fingerprint changed (input, engine, config, or notebook source). "
                "Move the old Step 2 working files aside before starting a fresh run."
            )
        if not ALL_PATH.exists():
            ALL_PATH.write_text("", encoding="utf-8")
        completed = [
            json.loads(line)
            for line in ALL_PATH.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        done_ids = [str(row["position_id"]) for row in completed]
        if len(done_ids) != len(set(done_ids)) or not set(done_ids).issubset(position_ids):
            raise ValueError("analysed_all.jsonl contains duplicate or unexpected position IDs.")
    else:
        state = {"fingerprint": fingerprint}
        write_json_atomic(STATE_PATH, state)
        ALL_PATH.write_text("", encoding="utf-8")
        completed = []
        done_ids = []

    done_id_set = set(done_ids)
    pending = [
        candidate
        for candidate in candidates
        if str(candidate["position_id"]) not in done_id_set
    ]
    engine = chess.engine.SimpleEngine.popen_uci(str(stockfish))
    try:
        engine.configure(
            {"Threads": config.STOCKFISH_THREADS, "Hash": config.STOCKFISH_HASH_MB}
        )
        engine_name = engine.id.get("name", "unknown")
        if f"stockfish {config.STOCKFISH_VERSION}".lower() not in engine_name.lower():
            raise RuntimeError(
                f"Configured Stockfish {config.STOCKFISH_VERSION}, but the engine reports {engine_name!r}."
            )
        state.update(engine_name=engine_name)
        write_json_atomic(STATE_PATH, state)
        if pending:
            print(f"Stockfish {engine_name}: {len(pending)} positions to analyse.")
        with ALL_PATH.open("a", encoding="utf-8", newline="\n") as output:
            for index, candidate in enumerate(pending, 1):
                row = analyse_position(engine, candidate)
                output.write(json.dumps(row, ensure_ascii=False) + "\n")
                output.flush()
                if index % 25 == 0:
                    print(f"Analysed {index:,}/{len(pending):,} pending positions.", flush=True)
    finally:
        engine.quit()

    analysed = []
    with ALL_PATH.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                analysed.append(json.loads(line))
    if len(analysed) != len(position_ids):
        raise RuntimeError(
            f"Expected {len(position_ids)} analysed positions, found {len(analysed)}."
        )

    kept = [row for row in analysed if not row["rejected_reasons"]]
    rejected = [row for row in analysed if row["rejected_reasons"]]
    pd.DataFrame(csv_rows(kept)).to_csv(
        KEPT_PATH, index=False, lineterminator="\n", float_format="%.10g"
    )
    pd.DataFrame(csv_rows(rejected)).to_csv(
        REJECTED_PATH, index=False, lineterminator="\n", float_format="%.10g"
    )
    reasons = Counter(
        reason for row in rejected for reason in row["rejected_reasons"]
    )
    kpis = dict(
        input_positions=len(candidates),
        analysed_positions=len(analysed),
        kept_positions=len(kept),
        rejected_positions=len(rejected),
        kept_by_kind=dict(Counter(row["kind"] for row in kept)),
        rejected_by_reason=dict(reasons),
    )
    KPI_PATH.write_text(json.dumps(kpis, indent=2), encoding="utf-8")

    hashed_outputs = sorted(
        [ALL_PATH.name, KEPT_PATH.name, REJECTED_PATH.name, KPI_PATH.name]
    )
    output_hash = hashlib.sha256()
    for name in hashed_outputs:
        output_hash.update(name.encode("utf-8"))
        output_hash.update((OUT / name).read_bytes())

    manifest = dict(
        step="step2_stockfish_recheck",
        code_version=CODE_VERSION,
        month=MONTH,
        timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        step1_results_sha256=step1_manifest["results_sha256"],
        pilot_split_complete=bool(step1_manifest.get("pilot_split_complete")),
        input_sha256=input_sha256,
        raw_sha256=step1_manifest["raw_sha256"],
        engine_name=engine_name,
        engine_sha256=engine_sha256,
        stockfish_version=config.STOCKFISH_VERSION,
        depth=config.STOCKFISH_DEPTH,
        threads=config.STOCKFISH_THREADS,
        hash_mb=config.STOCKFISH_HASH_MB,
        multipv=config.STOCKFISH_MULTIPV,
        source_sha256=STEP2_SOURCE_SHA256,
        config=config.config_snapshot(),
        git_commit=git_commit,
        git_uncommitted_changes=bool(git_status),
        python=sys.version.split()[0],
        platform=platform.platform(),
        packages={
            name: metadata.version(name)
            for name in ("chess", "numpy", "pandas")
        },
        kpis=kpis,
        hashed_outputs=hashed_outputs,
        results_sha256=output_hash.hexdigest(),
    )
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    state.update(completed=True, analyzed_positions=len(analysed))
    write_json_atomic(STATE_PATH, state)
    print(
        f"{len(analysed)} analysed -> {len(kept)} kept; "
        f"results_sha256={manifest['results_sha256']}"
    )


### Run Step 2


In [13]:
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
RUN_STOCKFISH = True  # Set True after Step 1 is complete and Stockfish is installed/configured.
if RUN_STOCKFISH: run_stockfish_recheck()
else: print("Step 2 skipped. Set RUN_STOCKFISH = True and run this cell when ready.")


506 analysed -> 446 kept; results_sha256=fe0249c236075523393cd9b03532678368a7225e09305e4a56fd2f74a68796b8
